In [ ]:
import timm # huggingface library
import torch
import torch.nn as nn

# 'efficientnet_b0' is the smallest standard variant
# pretrained=True downloads weights optimized on ImageNet
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=2)

# Verify the output features of the last layer
print(f"Model: {model.default_cfg['architecture']}")
print(f"Classifier head: {model.get_classifier()}")

Model: efficientnet_b0
Classifier head: Linear(in_features=1280, out_features=2, bias=True)


In [2]:
from torchvision import datasets
from torch.utils.data import DataLoader

# Get model-specific config (normalization, resize)
config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**config, is_training=True)

# Create loaders
train_ds = datasets.ImageFolder("PrepData/Training", transform=transform)
val_ds = datasets.ImageFolder("PrepData/Validation", transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [5]:
device

device(type='cuda')

In [4]:
from tqdm.auto import tqdm

epochs = 3
for epoch in range(epochs):
    # Training Phase
    model.train()
    train_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Simple Validation Phase
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / len(val_ds)
    print(f"Epoch {epoch+1}: Loss = {train_loss/len(train_loader):.4f}, Val Acc = {accuracy:.2f}%")

Epoch 1/3:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 1: Loss = 0.8247, Val Acc = 67.86%


Epoch 2/3:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 2: Loss = 0.3155, Val Acc = 96.22%


Epoch 3/3:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 3: Loss = 0.1894, Val Acc = 95.38%


In [6]:
torch.save(model.state_dict(), "fruit_classifier_b0.pth")